In [72]:
import os
os.chdir(r"C:\vscode\graph-rag\Source")

In [73]:
from config.settings_loader import load_config

config = load_config("config/config.yaml")

In [74]:
import pdfplumber

text = []
with pdfplumber.open(config["data_source"]["raw_data"]["pdf_path"]) as pdf:
    for page in pdf.pages:
        page_text = page.extract_text()
        if page_text:
            text.append(page_text)

raw_text = "\n\n".join(text)

In [75]:
with open(config["data_source"]["normalized_data"]["unstructured_text_path"], "w", encoding="utf-8") as f:
    f.write(raw_text)

In [ ]:
import re

#to add ## and # for chapters and parts
def add_md_headings(text: str) -> str:
    lines = text.splitlines()
    out = []
    i = 0

    while i < len(lines):
        line = lines[i].strip()

        # ---- detect chapter start ----
        if re.match(r"^Chapter\s+[IVXLC]+:", line):
            chapter_lines = [line]

            i += 1
            # collect continuation lines (title spillover)
            while i < len(lines):
                next_line = lines[i].strip()
                if not next_line:
                    break
                if re.match(r"^Part\s+[IVXLC]+\.?", next_line):
                    break
                if next_line.startswith("#"):
                    break
                # heuristic: titles are short and not sentence-ended
                if len(next_line) < 120 and not re.search(r"[.!?]$", next_line):
                    chapter_lines.append(next_line)
                    i += 1
                else:
                    break

            chapter_text = " ".join(chapter_lines).strip()
            out.append(f"## {chapter_text}")

            # ---- optional part line ----
            if i < len(lines):
                part_line = lines[i].strip()
                if re.match(r"^Part\s+[IVXLC]+\.?", part_line):
                    out.append(f"### {part_line}")
                    i += 1

            continue

        # ---- standalone part (rare but possible) ----
        if re.match(r"^Part\s+[IVXLC]+\.?", line):
            out.append(f"### {line}")
            i += 1
            continue

        # ---- normal line ----
        out.append(lines[i])
        i += 1

    return "\n".join(out)

#to remove the asterisks from the text
def remove_unnecessary_asterisks(text: str) -> str:
    # Remove "* Note:" or "* note:" (editorial markers)
    text = re.sub(r"\*\s*note\s*:", "", text, flags=re.IGNORECASE)

    # Remove inline sequences of 2+ asterisks (spaced or unspaced)
    text = re.sub(r"(\*\s*){2,}", "", text)

    cleaned_lines = []
    for line in text.splitlines():
        stripped = line.strip()

        # Remove lines containing only asterisks (any spacing)
        if re.fullmatch(r"(\*\s*)+", stripped):
            continue

        cleaned_lines.append(line)

    return "\n".join(cleaned_lines)

#to remove square bracket footnotes
def remove_footnotes_and_editorial(text: str) -> str:
    lines = text.splitlines()
    cleaned = []

    in_footnote_block = False

    for line in lines:
        stripped = line.strip()

        # ---- Drop inline numeric markers like "1a", "2c", "5" ----
        line = re.sub(r"\s\d+[a-z]?\b", "", line)

        # ---- Detect start of footnote/editorial blocks ----
        if (
            stripped.startswith("[")
            or stripped.startswith("Note:")
            or re.match(r"^\d+\s*\(return\)", stripped)
        ):
            in_footnote_block = True
            continue

        # ---- Detect end of editorial blocks ----
        if in_footnote_block:
            if stripped.endswith("—G.]") or stripped.endswith("—M.]") \
               or stripped.endswith("—G.") or stripped.endswith("—M.]") \
               or stripped.endswith("]"):
                in_footnote_block = False
            continue

        # ---- Skip empty lines created by removals ----
        if not stripped:
            if cleaned and cleaned[-1] != "":
                cleaned.append("")
            continue

        cleaned.append(line.rstrip())

    # Normalize excess blank lines
    output = "\n".join(cleaned)
    output = re.sub(r"\n{3,}", "\n\n", output)

    return output.strip()

#removes only editorial / footnote numbers
def remove_editorial_numbers(text: str) -> str:
    lines = text.splitlines()
    cleaned = []

    for line in lines:
        stripped = line.strip()

        # Remove lines that are ONLY footnote backlinks
        # e.g. "2c (return)", "3a (return)"
        if re.fullmatch(r"\d+[a-z]?\s*\(return\)", stripped):
            continue

        # Remove leading editorial numbers at paragraph start
        # e.g. "15 Decebalus, the Dacian king..."
        line = re.sub(r"^\s*\d+[a-z]?\s+(?=[A-Z])", "", line)

        cleaned.append(line)

    return "\n".join(cleaned)


# for reconstructing the lines
def fix_line_wraps(text: str) -> str:
    lines = text.splitlines()
    output = []
    buffer = ""

    def is_heading(line: str) -> bool:
        return line.startswith("#")

    def looks_like_title(line: str) -> bool:
        # short, title-cased lines without punctuation
        return (
            len(line) < 80
            and not re.search(r"[.!?]$", line)
            and line == line.title()
        )

    for line in lines:
        stripped = line.strip()

        # --- hard paragraph break ---
        if not stripped:
            if buffer:
                output.append(buffer.strip())
                buffer = ""
            output.append("")
            continue

        # --- headings stay isolated ---
        if is_heading(stripped):
            if buffer:
                output.append(buffer.strip())
                buffer = ""
            output.append(stripped)
            continue

        # --- section titles stay isolated ---
        if looks_like_title(stripped):
            if buffer:
                output.append(buffer.strip())
                buffer = ""
            output.append(stripped)
            continue

        # --- line joining logic ---
        if buffer:
            # if buffer already ends a sentence, flush it
            if re.search(r"[.!?]$", buffer):
                output.append(buffer.strip())
                buffer = stripped
            else:
                buffer += " " + stripped
        else:
            buffer = stripped

    if buffer:
        output.append(buffer.strip())

    # normalize extra blank lines
    text = "\n".join(output)
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text

#reconstruction of paragraph
def reconstruct_paragraphs(text: str) -> str:
    paragraphs = [p.strip() for p in text.split("\n\n") if p.strip()]
    rebuilt = []
    buffer = ""

    def ends_with_sentence(p: str) -> bool:
        return bool(re.search(r"[.!?]$", p.strip()))

    for para in paragraphs:
        if not buffer:
            buffer = para
        else:
            if not ends_with_sentence(buffer):
                buffer = buffer + " " + para
            else:
                rebuilt.append(buffer.strip())
                buffer = para

    if buffer:
        # ensure paragraph ends cleanly
        if not ends_with_sentence(buffer):
            buffer = buffer.rstrip() + "."
        rebuilt.append(buffer.strip())

    return "\n\n".join(rebuilt)

In [81]:
# read normalized text
with open(
    config["data_source"]["normalized_data"]["unstructured_text_path"],
    "r",
    encoding="utf-8"
) as f:
    structured_data = f.read()

# add markdown headings
structured_data = add_md_headings(structured_data)
structured_data = remove_unnecessary_asterisks(structured_data)
structured_data = remove_footnotes_and_editorial(structured_data)
structured_data = remove_editorial_numbers(structured_data)
structured_data = fix_line_wraps(structured_data)
structured_data = reconstruct_paragraphs(structured_data)

# write to structured location
with open(
    config["data_source"]["normalized_data"]["structured_text_path"],
    "w",
    encoding="utf-8"
) as f:
    f.write(structured_data)